---
title: "Fifteen Good Cards"
author: "Bunburyist"
date: today
format:
  html:
    theme: cosmo          # clean Bootstrap theme
    toc: true             # table of contents
    toc-depth: 3          # how many heading levels to include
    number-sections: true # numbered section headings
    code-fold: true       # allow toggling code visibility
    df-print: paged       # paginated DataFrames
execute:
  echo: false   # hide code by default (can override per-cell)
  warning: false
  message: false
---

# 15 Good Cards

This is a numerical analysis of [Earthborne Rangers](https://thelivingvalley.earthbornegames.com) decks posted on [RangersDB](https://rangersdb.com/decks/search). Our objective is to use wisdom of the masses to identify the most popular cards for different Backgrounds and Specialties, weighted toward the more influential decks, with the ultimate goal of suggesting cards for various decks.

In [138]:
from itertools import permutations
import pandas as pd
import pickle

In [ ]:
#| include: false
with open('all_cards.pkl', 'rb') as f:
  all_cards = pickle.load(f)
df_cards = pd.DataFrame(all_cards)
print(f'Loaded {df_cards.shape[0]} cards')

Loaded 265 cards


In [178]:
#| include: false
# Define pack order
PACK_ORDER = [
  'core',  # Core set
  'sotv',  # Stewards of the Valley
  'loa'    # Legacy of the Ancestors
]
PACK_TO_INDEX = {pack_id: i for i, pack_id in enumerate(PACK_ORDER)}
INDEX_TO_PACK = {i: pack_id for i, pack_id in enumerate(PACK_ORDER)}
df_cards["card_pack_index"] = df_cards["pack_id"].map(PACK_TO_INDEX)
card_id_to_pack_index = (df_cards
  .set_index("id")
  .to_dict()["card_pack_index"])
card_id_to_name = (df_cards
  .set_index("id")
  .to_dict()["name"])
print('Count cards in each pack:')
df_cards.groupby("card_pack_index").size()

Count cards in each pack:


card_pack_index
0    155
1     81
2     29
dtype: int64

In [55]:
#| include: false
# These cards id were enhanced in taboo set_01, so if the user liked the older
# version, they would also like the newer version
NOT_NERFED = ('01041', '01044', '01069', '01074', '01084')
# taboo_id is None for the original version of each card
df_cards["taboo_fill_id"] = df_cards["taboo_id"].fillna('set_00')
df_cards["latest_taboo_id"] = (df_cards
  .groupby("code")["taboo_fill_id"]
  .transform("max"))
df_cards["is_latest"] = (
  (df_cards["taboo_fill_id"] == df_cards["latest_taboo_id"])
  | df_cards["id"].isin(NOT_NERFED))

print('Latest cards in each pack:')
df_cards.groupby("card_pack_index").agg(
  latest=("is_latest", "sum"),
  not_latest=("is_latest", lambda x: (~x).sum()))

Latest cards in each pack:


,latest,not_latest
card_pack_index,,
0,145,10
1,81,0
2,29,0


In [ ]:
#| include: false
# set_type_id and set_id don't quite do what we want, so modify them into
# supergroup_id and subgroup_id
def get_supergroup_id(card):
  if card["set_id"] == 'personality':
    return 'Personality'
  set_type_id = card["set_type_id"]
  if set_type_id == 'specialty' and card["type_id"] == 'role':
    return 'Role'
  return (set_type_id.capitalize()
          if isinstance(set_type_id, str) else set_type_id)

def get_subgroup_id(card):
  if card["set_id"] == "personality":
    return card["aspect_name"].capitalize()
  return card["set_id"].capitalize()

df_cards["supergroup_id"] = df_cards.apply(get_supergroup_id, axis=1)
df_cards["subgroup_id"] = df_cards.apply(get_subgroup_id, axis=1)

print('Supergroup and subgroup combinations:')
(df_cards[["supergroup_id", "subgroup_id"]]
  .drop_duplicates()
  .sort_values(by=["supergroup_id", "subgroup_id"]))

Supergroup and subgroup combinations:


,supergroup_id,subgroup_id
20,Background,Artisan
28,Background,Forager
10,Background,Shepherd
184,Background,Talespinner
11,Background,Traveler
63,Personality,Awareness
58,Personality,Fitness
61,Personality,Focus
62,Personality,Spirit
54,Role,Artificer


In [179]:
#| include: false
with open('decks.pkl', 'rb') as f:
  all_decks = pickle.load(f)
decks_clean = [deck for deck in all_decks.values() if deck is not None]
df_all_decks = pd.DataFrame(decks_clean)
print(f'Loaded {df_all_decks.shape[0]} decks')
df_all_decks["taboo_set_id"] = df_all_decks["taboo_set_id"].fillna('set_00')

Loaded 13002 decks


In [ ]:
#| include: false
# Move background, specialty, and role_id from meta to columns
def extract_background(meta):
  return meta.get("background", None)
def extract_specialty(meta):
  return meta.get("specialty", None)
def extract_role_id(meta):
  return meta.get("role", None)

df_all_decks["background"] = df_all_decks["meta"].apply(extract_background)
df_all_decks["specialty"] = df_all_decks["meta"].apply(extract_specialty)
df_all_decks["role_id"] = df_all_decks["meta"].apply(extract_role_id)

df_all_decks["is_ignore"] = (
  df_all_decks["background"].isna()
  | df_all_decks["specialty"].isna()
  | df_all_decks["role_id"].isna())
ignore_count = df_all_decks["is_ignore"].sum()
print(f'Ignored decks: {ignore_count}')

Ignored decks: 34


In [ ]:
#| include: false
# ignore pre-built decks
def card_name_to_ids(name):
  return df_cards.loc[
    (df_cards["name"] == name),
    "id"
  ].tolist()

UNDAUNTED_SEEKER = {
  'background': 'traveler',
  'specialty': 'explorer',
  'role_id': card_name_to_ids('Undaunted Seeker'),
  'awa': 2,
  'spi': 2,
  'fit': 3,
  'foc': 1,
  'slots': (
    card_name_to_ids('Meticulous'),
    card_name_to_ids('Determined'),
    card_name_to_ids('Insightful'),
    card_name_to_ids('Persuasive'),
    card_name_to_ids('Eagle Eye'),
    card_name_to_ids('Ironwool Boots'),
    card_name_to_ids('Perfect Recall'),
    card_name_to_ids('Reverb Locket'),
    card_name_to_ids('Strider'),
    card_name_to_ids('Afforded by Nature'),
    card_name_to_ids('A Leaf in the Breeze'),
    card_name_to_ids('Boundary Sensor'),
    card_name_to_ids('Orlin Hiking Stave'),
    card_name_to_ids('Walk With Me'),
    card_name_to_ids('Trail Markers'),
  )
}

VOICE_OF_THE_ELDERS = {
  'background': 'shepherd',
  'specialty': 'conciliator',
  'role_id': card_name_to_ids('Voice of the Elders'),
  'awa': 1,
  'spi': 3,
  'fit': 2,
  'foc': 2,
  'slots': (
    card_name_to_ids('Engaging'),
    card_name_to_ids('Astute'),
    card_name_to_ids('Passionate'),
    card_name_to_ids('Perceptive'),
    card_name_to_ids('Calming Presence'),
    card_name_to_ids('Healing Touch'),
    card_name_to_ids('Homeward Bound'),
    card_name_to_ids('Oru the Sheep Dog'),
    card_name_to_ids('Paratrepsis Whistle'),
    card_name_to_ids('A Dear Friend'),
    card_name_to_ids('Follow in Footsteps'),
    card_name_to_ids('Intention Translator'),
    card_name_to_ids('Pokodo the Ferret'),
    card_name_to_ids('Surveyed Land'),
    card_name_to_ids('Energized Hiking Greaves'),
  )
}

MASTERFUL_ENGINEER = {
  'background': 'artisan',
  'specialty': 'artificer',
  'role_id': card_name_to_ids('Masterful Engineer'),
  'awa': 2,
  'spi': 2,
  'fit': 1,
  'foc': 3,
  'slots': (
    card_name_to_ids('Balanced'),
    card_name_to_ids('Inventive'),
    card_name_to_ids('Thorough'),
    card_name_to_ids('Compassionate'),
    card_name_to_ids('Favorite Gear'),
    card_name_to_ids('Functional Replica'),
    card_name_to_ids('Moment of Desperation'),
    card_name_to_ids('Pocketed Belt Pouch'),
    card_name_to_ids('Universal Power Cells'),
    card_name_to_ids('A Stone in the River'),
    card_name_to_ids('Ferinodex'),
    card_name_to_ids('Infusion Canteen'),
    card_name_to_ids('Memorill Sketchpad'),
    card_name_to_ids('Memlev Trekking Poles'),
    card_name_to_ids('Adaptable Multitool'),
  )
}

ADHERENT_OF_THE_FIRST_IDEAL = {
  'background': 'forager',
  'specialty': 'shaper',
  'role_id': card_name_to_ids('Adherent of the First Ideal'),
  'awa': 3,
  'spi': 1,
  'fit': 2,
  'foc': 2,
  'slots': (
    card_name_to_ids('Bold'),
    card_name_to_ids('Thoughtful'),
    card_name_to_ids('Versatile'),
    card_name_to_ids('Vigilant'),
    card_name_to_ids('Familiar Ground'),
    card_name_to_ids('Green Thumb'),
    card_name_to_ids('Local Fare'),
    card_name_to_ids('Loose-Leaf Tea Kit'),
    card_name_to_ids("Nature's Abundance"),
    card_name_to_ids('Root Snare'),
    card_name_to_ids('Shape the Earth'),
    card_name_to_ids('Sky Whip'),
    card_name_to_ids('Stave of the Sun'),
    card_name_to_ids('What Should Never Be'),
    card_name_to_ids('Harmonize'),
  )
}

CARTOGRAPHER_OF_MANY_WORLDS = {
  'background': 'talespinner',
  'specialty': 'spirit_speaker',
  'role_id': card_name_to_ids('Cartographer of Many Worlds'),
  'awa': 1,
  'spi': 2,
  'fit': 2,
  'foc': 3,
  'slots': (
    card_name_to_ids('Dauntless'),
    card_name_to_ids('Enthusiastic'),
    card_name_to_ids('Ingenious'),
    card_name_to_ids('Sensible'),
    card_name_to_ids('Wonders of the Valley'),
    card_name_to_ids('The Buck and the Wolhund'),
    card_name_to_ids('Book of Great Deeds'),
    card_name_to_ids('The Tireless Artisan'),
    card_name_to_ids('Songweave Mantle'),
    card_name_to_ids('Irix Spirit'),
    card_name_to_ids('The Path Opens Before Me'),
    card_name_to_ids('Atrox Spirit'),
    card_name_to_ids('The Long Walk'),
    card_name_to_ids('Lutrinal Spirit'),
    card_name_to_ids('Piercing Whistle'),
  )
}

def if_match_deck(deck, premade):
  for key, value in premade.items():
    if key == 'role_id':
      if deck['role_id'] not in value:
        return False

    elif key == 'slots':
      flatten_slots = [id for id_list in value for id in id_list]
      for card_id in deck['slots']:
        if card_id not in flatten_slots:
          return False

    elif deck[key] != value:
      return False

  return True

df_all_decks["is_ignore"] = (
  df_all_decks["is_ignore"]
  | df_all_decks.apply(lambda deck: if_match_deck(deck, UNDAUNTED_SEEKER), axis=1)
  | df_all_decks.apply(lambda deck: if_match_deck(deck, VOICE_OF_THE_ELDERS), axis=1)
  | df_all_decks.apply(lambda deck: if_match_deck(deck, MASTERFUL_ENGINEER), axis=1)
  | df_all_decks.apply(lambda deck: if_match_deck(deck, ADHERENT_OF_THE_FIRST_IDEAL), axis=1)
  | df_all_decks.apply(lambda deck: if_match_deck(deck, CARTOGRAPHER_OF_MANY_WORLDS), axis=1)
)
ignore_count = df_all_decks["is_ignore"].sum()
print(f'\nIgnored decks: {ignore_count}')


Ignored decks: 817


In [ ]:
#| include: false
# For each deck:
# - Set deck_pack_index to the highest pack index among its cards
# - Calculate the equip total among its cards
# - Ignore it if it doesn't have 15 cards or has a reward or malady
def check_cards(deck):
  slots = [card_id for card_id in deck["slots"].keys()]
  has_15_cards = (len(slots) == 15)
  # decks have a 16th card, their role card
  slots.append(deck["role_id"])
  cards = df_cards[df_cards["id"].isin(slots)]
  deck_pack_index = cards["card_pack_index"].max()
  equip_sum = cards["equip"].sum()
  has_spoiler = cards["spoiler"].any()
  is_latest = cards["is_latest"].all()
  return pd.Series({
    "deck_pack_index": deck_pack_index,
    "equip_sum": equip_sum,
    "has_15_cards": has_15_cards,
    "has_spoiler": has_spoiler,
    "is_latest": is_latest
  })

card_check = df_all_decks.apply(check_cards, axis=1)
# Drop columns so running this cell repeatedly doesn't throw an error
for col in card_check.columns:
  if col in df_all_decks.columns:
    print(f'Dropping column {col}')
    df_all_decks = df_all_decks.drop(columns=[col])
df_all_decks = pd.concat([df_all_decks, card_check], axis=1)
df_all_decks["deck_pack_index"] = (df_all_decks["deck_pack_index"]
  .astype('Int64'))

print(f"\ndeck_pack_index distribution:")
for deck_pack_index in range(len(PACK_ORDER)):
  pack_id = INDEX_TO_PACK[deck_pack_index]
  count = (df_all_decks["deck_pack_index"] == deck_pack_index).sum()
  print(f"  {pack_id} (index {deck_pack_index}): {count} decks")

df_all_decks["is_ignore"] = (
  df_all_decks["is_ignore"]
  | ~df_all_decks["has_15_cards"]
  | df_all_decks["has_spoiler"]
  # | df_all_decks["is_latest"]
)
ignore_count = df_all_decks["is_ignore"].sum()
print(f'\nIgnored decks: {ignore_count}')


deck_pack_index distribution:
  core (index 0): 10313 decks
  sotv (index 1): 2537 decks
  loa (index 2): 148 decks

Ignored decks: 4902


In [ ]:
#| include: false
# Find each deck's immediate parent
def get_all_decks_parent_id(all_decks_row):
  if all_decks_row["previous_deck"] is not None:
    previous_id = all_decks_row["previous_deck"]["id"]
    # Sometimes the previous deck has been deleted
    if previous_id in df_all_decks["id"].values:
      return previous_id

  if all_decks_row["original_deck"] is not None:
    original_id = all_decks_row["original_deck"]["deck"].get("id", None)
    if original_id in df_all_decks["id"].values:
      return original_id

  return None

df_all_decks["all_decks_parent_id"] = (df_all_decks
  .apply(get_all_decks_parent_id, axis=1)
  .astype('Int64'))
assert (set(df_all_decks["all_decks_parent_id"].dropna())
  <= set(df_all_decks["id"]))
num_parent = df_all_decks["all_decks_parent_id"].notna().sum()
num_all_decks = df_all_decks.shape[0]
print(f'Number of parent_id: {num_parent} out of {num_all_decks} decks')

Number of parent_id: 4002 out of 13002 decks


In [ ]:
#| include: false
# df_decks contains only non-ignored decks
df_decks = df_all_decks[~df_all_decks["is_ignore"]].copy()
print(f'{df_decks.shape[0]} non-ignored decks')

8100 non-ignored decks


In [ ]:
#| include: false
# Find each deck's closest non-ignored ancestor
def get_parent_id(deck):
  current_id = deck["all_decks_parent_id"]
  while pd.notna(current_id):
    if current_id in df_decks["id"].values:
      return current_id
      
    current_deck = df_all_decks[df_all_decks["id"] == current_id].iloc[0]
    current_id = current_deck["all_decks_parent_id"]

  return None

df_decks["parent_id"] = (df_decks
  .apply(get_parent_id, axis=1)
  .astype('Int64'))
assert set(df_decks["parent_id"].dropna()) <= set(df_decks["id"])
num_parent = df_decks["parent_id"].notna().sum()
num_decks = df_decks.shape[0]
print(f'Number of parent_id: {num_parent} out of {num_decks} decks')

Number of parent_id: 1022 out of 8100 decks


In [ ]:
#| include: false
# Find each deck's closest non-ignored ancestor from a different user
def get_inspiration_id(deck):
  this_user_id = deck["user_id"]
  current_id = deck["parent_id"]
  while pd.notna(current_id):
    current_deck = df_decks[df_decks["id"] == current_id].iloc[0]
    current_user_id = current_deck["user_id"]
    if current_user_id != this_user_id:
      return current_id
      
    current_id = current_deck["parent_id"]

  return None

df_decks["inspiration_id"] = (df_decks
  .apply(get_inspiration_id, axis=1)
  .astype('Int64'))
num_inspiration = df_decks["inspiration_id"].notna().sum()
num_decks = df_decks.shape[0]
print(f'Number of inspiration_id: {num_inspiration} out of {num_decks} decks')

Number of inspiration_id: 526 out of 8100 decks


In [156]:
#| include: false
# Identify decks with non-zero user_weight
user_parent_ids = (df_decks.groupby("user_id")["parent_id"]
  .apply(lambda parents: set(parents.dropna())))
df_decks["user_weight"] = (df_decks
  .apply(lambda row: row["id"] not in user_parent_ids[row["user_id"]], axis=1)
  .astype('Int64'))

# Divide 1 user_weight among (user_id, background, specialty) decks
sum_weights = (df_decks
  .groupby(["user_id", "background", "specialty"])["user_weight"]
  .transform("sum"))
df_decks["user_weight"] = (df_decks["user_weight"]
  / sum_weights.where(sum_weights > 0, 1))

user_weight_sum = df_decks["user_weight"].sum()
user_weight_size = (df_decks["user_weight"] > 0).size
print(f'{user_weight_sum} total user_weight split among {user_weight_size} decks')

5700.0 total user_weight split among 8100 decks


In [157]:
#| include: false
# Decks with no inspiration keep their full user_weight
# Decks with inspiration donate half their user_weight
from_self = df_decks.apply(lambda row: 0.5 * row["user_weight"]
  if pd.notna(row["inspiration_id"]) else row["user_weight"], axis=1)

# Decks receive half the user_weight of decks they inspired
df_inspired = df_decks[df_decks["inspiration_id"].notna()]
sum_weights = df_inspired.groupby("inspiration_id")["user_weight"].sum()
from_inspired = df_decks["id"].map(lambda x: 0.5 * sum_weights.get(x, 0))

df_decks["deck_weight"] = from_self + from_inspired

deck_weight_sum = df_decks["deck_weight"].sum()
deck_weight_size = (df_decks["deck_weight"] > 0).size
print(f'{deck_weight_sum} total deck_weight split among {deck_weight_size} decks')

5700.0 total deck_weight split among 8100 decks


In [ ]:
#| include: false
# Calculate how many decks have adopted taboo
taboo_counts = {taboo_id: (df_decks["taboo_set_id"] <= taboo_id).sum()
  for taboo_id in df_decks["taboo_set_id"].unique()}
max_taboo_count = max(taboo_counts.values())
map_taboo_weight = {taboo_id: float(count / max_taboo_count)
  for taboo_id, count in taboo_counts.items()}
df_decks["taboo_weight"] = df_decks["taboo_set_id"].map(map_taboo_weight)
print(f"Taboo weights: {map_taboo_weight}")
df_decks["group_score"] = df_decks["deck_weight"] * df_decks["taboo_weight"]

Taboo weights: {'set_00': 0.8955555555555555, 'set_01': 1.0}


We are hiding the math, but we tried to factory in the following considerations:

- Players who build decks drawing from Core set and Stewards of the Valley (SotV) consider more cards than those who build from the Core set alone.
- However, there are fewer decks that pick from the Core set plus SotV than the Core set alone because the SotV has not been out as long and fewer people have SotV than the Core set.
- Players make new versions of existing decks. Furthermore, a player who modifies another player's deck suggests they were inspired by the original deck, meanwhile a player modifying their own deck indicates they aren't satisfied with their original deck.
- Some players use the [Premade Ranger Decks](https://earthbornegames.com/resources/ranger-decks).
- [The Elder's Book of Uncommon Wisdom](https://thelivingvalley.earthbornegames.com/docs/updates/elders_book_of_uncommon_wisdom/) has updated some cards, but adoption of the tEBoUW has been low.

Here are the most popular combinations of Background and Specialty (according to my definition of "popular"):

In [159]:
def group_and_score(df, groupby_cols, score_col, pack_index_col):
  # for each (group, pack), total score
  df_group_pack = (df
    .groupby(groupby_cols + [pack_index_col])["group_score"]
    .sum()
    .reset_index())
  # for each pack, average score per group
  df_pack_map = (df_group_pack
    .groupby(pack_index_col)["group_score"]
    .mean()
    .to_dict())
  # for each (group, pack), add average score per group based on pack
  df_group_pack["pack_avg"] = df_group_pack[pack_index_col].map(df_pack_map)
  # for each group, total score and average score over all its packs
  df_group = (df_group_pack
    .groupby(groupby_cols)[["group_score", "pack_avg"]]
    .sum()
    .reset_index())
  # compare group's score to average group's score
  df_group["score"] = df_group["group_score"] / df_group["pack_avg"]
  return df_group.sort_values("score", ascending=False)

In [ ]:
bg_sp_score = group_and_score(df_decks,
  groupby_cols=["background", "specialty"],
  score_col="group_score",
  pack_index_col="deck_pack_index")

bg_sp_score = bg_sp_score.rename(columns={
  'background': 'Background',
  'specialty': 'Specialty',
  'score': 'Popularity'
})
bg_sp_score["Background"] = bg_sp_score["Background"].str.title()
bg_sp_score["Specialty"] = bg_sp_score["Specialty"].apply(
  lambda s: s.replace('_', ' ').title()
)
bg_sp_score[["Background", "Specialty", "Popularity"]]

,Background,Specialty,Popularity
11,Shepherd,Conciliator,2.586443
0,Artisan,Artificer,1.940267
22,Traveler,Explorer,1.923161
18,Talespinner,Shaper,1.781636
23,Traveler,Shaper,1.584909
14,Shepherd,Spirit Speaker,1.429241
19,Talespinner,Spirit Speaker,1.291304
8,Forager,Shaper,1.219189
16,Talespinner,Conciliator,1.130537
7,Forager,Explorer,1.068615


In [161]:
def group_and_score_wrapper(df, groupby_column, score_col, pack_index_col,
    num_results=None):
  # for cards with multiple ids, we group them together by name
  role_scores = group_and_score(df,
    groupby_cols=[groupby_column],
    score_col=score_col,
    pack_index_col=pack_index_col)

  if num_results is None:
    num_results = len(role_scores)

  print(f"{'Name':<35} {'Group':<6} / {'Avg':<6} = {'Score':<6}")
  print('-' * 60)
  for _, row in role_scores.head(num_results).iterrows():
    name = row[groupby_column]
    group_sum = row["group_score"]
    avg_sum = row["pack_avg"]
    score = row["score"]
    print(f"{name:<35} {group_sum:<6.1f} / {avg_sum:<6.1f} = {score:<6.3f}")

def extract_cards(df, score_col, pack_index_col):
  # attach score and pack_index of deck to each card
  bg_sp_slots_data = []
  for _, deck in df.iterrows():
    slots = deck["slots"]
    for card_id, quantity in slots.items():
      card_name = card_id_to_name[card_id]
      bg_sp_slots_data.append({
        "name": card_name,
        score_col: deck[score_col],
        pack_index_col: deck[pack_index_col]
      })

  # we don't need individual cards, just their sum
  bg_sp_slots_long = pd.DataFrame(bg_sp_slots_data)
  bg_sp_slots = (
    bg_sp_slots_long.groupby(["name", pack_index_col])
    [score_col].sum()
    .reset_index()
    .astype({pack_index_col: "Int64", score_col: "float64"})
  )

  # bg_sp_slots may not contain all allowed outside interest cards
  # get all cards that users can build their decks with
  available_cards = (df_cards[
    df_cards["supergroup_id"].isin(['Personality', 'Background', 'Specialty'])
      # & df_cards["is_latest"]
    ][["name", "supergroup_id", "subgroup_id"]]
    .drop_duplicates()
    .copy())
  # distinguish outside interest cards from background and specialty cards
  outside_interest_mask = (
    (available_cards["supergroup_id"] == 'Background')
      & (available_cards["subgroup_id"] != background.capitalize())
    ) | (
    (available_cards["supergroup_id"] == 'Specialty')
      & (available_cards["subgroup_id"] != specialty.capitalize()))
  available_cards.loc[outside_interest_mask, "supergroup_id"] = 'Outside'
  available_cards.loc[outside_interest_mask, "subgroup_id"] = 'Interest'

  # join all cards with card data from decks
  bg_sp_cards = (available_cards
    .merge(
      bg_sp_slots,
      how='left',
      on="name")
    .astype({pack_index_col: "Int64", score_col: "float64"})
    .fillna(0)
  )
  return bg_sp_cards

def score_background_specialty(background, specialty, score_col, pack_index_col):
  bg_sp_decks = df_decks[(df_decks["background"] == background)
    & (df_decks["specialty"] == specialty)
    & (df_decks[score_col] > 0)].copy()

  bg_sp_decks["role"] = bg_sp_decks["role_id"].map(card_id_to_name)
  print('Role scores:')
  group_and_score_wrapper(bg_sp_decks, "role", score_col, pack_index_col)

  bg_sp_cards = extract_cards(bg_sp_decks, score_col, pack_index_col)

  build_order = [('Personality', 'Awareness', 3),
                ('Personality', 'Fitness', 3),
                ('Personality', 'Focus', 3),
                ('Personality', 'Spirit', 3),
                ('Background', background.capitalize(), 10),
                ('Specialty', specialty.capitalize(), 10),
                ('Outside', 'Interest', 10)]

  for supergroup_id, subgroup_id, num_results in build_order:
    build_cards = (bg_sp_cards[
      (bg_sp_cards["supergroup_id"] == supergroup_id)
      & (bg_sp_cards["subgroup_id"] == subgroup_id)]
      .copy())
      
    print(f'\n{supergroup_id} {subgroup_id} scores:')
    group_and_score_wrapper(build_cards, "name", score_col, pack_index_col,
      num_results)

In [162]:
background = 'shepherd'
specialty = 'conciliator'
score_background_specialty(background, specialty,
  score_col="group_score",
  pack_index_col="deck_pack_index")

Role scores:
Name                                Group  / Avg    = Score 
------------------------------------------------------------
Voice of the Elders                 411.0  / 366.8  = 1.120 
Guardian                            337.3  / 366.8  = 0.920 
Pathwarden                          22.1   / 36.8   = 0.602 

Personality Awareness scores:
Name                                Group  / Avg    = Score 
------------------------------------------------------------
Perceptive                          295.0  / 183.4  = 1.608 
Insightful                          237.9  / 183.4  = 1.297 
Thorough                            187.6  / 183.4  = 1.023 

Personality Fitness scores:
Name                                Group  / Avg    = Score 
------------------------------------------------------------
Bold                                349.4  / 183.4  = 1.905 
Balanced                            178.7  / 183.4  = 0.974 
Passionate                          162.2  / 183.4  = 0.884 

Personality

In [125]:
background = 'traveler'
specialty = 'artificer'
score_background_specialty(background, specialty,
  score_col="group_score",
  pack_index_col="deck_pack_index")

Role scores:
Name                                Group  / Avg    = Score 
------------------------------------------------------------
Exceptional Tinkerer                125.8  / 74.3   = 1.695 
Gregarious Inventor                 4.6    / 7.7    = 0.593 
Masterful Engineer                  25.8   / 74.3   = 0.347 

Personality Awareness scores:
Name                                Group  / Avg    = Score 
------------------------------------------------------------
Thorough                            64.9   / 32.4   = 2.004 
Perceptive                          53.8   / 32.4   = 1.660 
Insightful                          19.8   / 32.4   = 0.612 

Personality Fitness scores:
Name                                Group  / Avg    = Score 
------------------------------------------------------------
Balanced                            50.4   / 37.1   = 1.358 
Passionate                          48.8   / 37.1   = 1.316 
Steadfast                           4.1    / 3.9    = 1.061 

Personality

In [164]:
background = 'shepherd'
specialty = 'conciliator'
df_bg_sp = df_decks[
  (df_decks["background"] == background)
  & (df_decks["specialty"] == specialty)
]

ASPECTS = ['awa', 'fit', 'foc', 'spi']
aspect_scores = {}
for aspect in ASPECTS:
  aspect_scores[aspect] = (
    df_bg_sp.groupby(aspect)["deck_weight"]
    .sum()
    .to_dict()
  )

aspect_lines = (
  set(permutations((1, 2, 2, 3)))
  .union(set(permutations((1, 1, 2, 4))))
)

max_score = None
best_aspect_line = None
for aspect_line in aspect_lines:
  score = sum([
    aspect_scores[aspect].get(value, 0)
    for aspect, value in zip(ASPECTS, aspect_line)
  ])
  if (max_score is None) or (score > max_score):
    max_score = score
    best_aspect_line = aspect_line

print("Best aspect_line:", best_aspect_line, "with score:", max_score)

Best aspect_line: (2, 2, 1, 3) with score: 2255.7107142857144


In [ ]:
equip_counts = df_bg_sp.groupby("equip_sum")["deck_weight"].sum()

# 1) equip_sum with the highest sum
max_equip_sum = equip_counts.idxmax()
max_equip_sum_value = equip_counts.max()
print(f"equip_sum with highest sum: {int(max_equip_sum)} (sum: {max_equip_sum_value})")

# 2) weighted average of equip_sum weighted by its sum
equip_counts = equip_counts.reset_index()
weighted_avg_equip_sum = (
  (
    equip_counts["equip_sum"]
    * equip_counts["deck_weight"]
  ).sum()
  / equip_counts["deck_weight"].sum())
print(f"Weighted average of equip_sum: {weighted_avg_equip_sum:.3f}")

equip_sum with highest sum: 5 (sum: 341.584126984127)
Weighted average of equip_sum: 4.092


In [134]:
for aspect in ['awa', 'fit', 'foc', 'spi']:
  sums = (
    df_decks.groupby(["background", "specialty", aspect], as_index=False)["deck_weight"]
    .sum()
  )
  top = (
    sums.sort_values("deck_weight", ascending=False)
    .drop_duplicates(["background", "specialty"])
  )
  pivot = top.pivot(index="background", columns="specialty", values=aspect)
  print(f"\nMost common {aspect}:")
  print(pivot)


Most common awa:
specialty    artificer  conciliator  explorer  shaper  spirit_speaker
background                                                           
artisan              2            2         2       2               1
forager              2            2         2       2               1
shepherd             2            2         2       2               1
talespinner          2            2         1       1               1
traveler             2            2         2       2               1

Most common fit:
specialty    artificer  conciliator  explorer  shaper  spirit_speaker
background                                                           
artisan              2            2         2       2               2
forager              2            2         2       2               2
shepherd             2            2         2       2               2
talespinner          1            2         2       2               2
traveler             2            2         3       3 